In [5]:
!apt-get install -y cd-hit -q

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  cd-hit
0 upgraded, 1 newly installed, 0 to remove and 24 not upgraded.
Need to get 521 kB of archives.
After this operation, 1,082 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 cd-hit amd64 4.8.1-4 [521 kB]
Fetched 521 kB in 2s (290 kB/s)
Selecting previously unselected package cd-hit.
(Reading database ... 118332 files and directories currently installed.)
Preparing to unpack .../cd-hit_4.8.1-4_amd64.deb ...
Unpacking cd-hit (4.8.1-4) ...
Setting up cd-hit (4.8.1-4) ...
Processing triggers for man-db (2.10.2-1) ...


In [6]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/AOP-ProCon"
!mkdir -p "{BASE}/data/raw" "{BASE}/data/processed" "{BASE}/data/embeddings" "{BASE}/results"

import pandas as pd
import numpy as np
import requests, re, subprocess

pd.set_option("display.max_colwidth", 80)

/bin/bash: line 1: nvidia-smi: command not found
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import pandas as pd, subprocess, re

aop_benchpos = pd.read_csv(f"{BASE}/data/processed/AOP-BenchPos.csv")

def cluster_40_safe(df, min_cluster_length=10):
    df = df.reset_index(drop=True)
    long_mask = df["sequence"].str.len() >= min_cluster_length
    short_mask = ~long_mask

    fasta_path = "/content/aop_benchpos_40_long.fasta"
    output_prefix = "/content/aop_benchpos_40_long"
    with open(fasta_path, "w") as f:
        for i in df[long_mask].index:
            f.write(f">seq{i}\n{df.loc[i, 'sequence']}\n")

    subprocess.run([
        "cd-hit", "-i", fasta_path, "-o", output_prefix,
        "-c", "0.40", "-n", "2", "-l", str(min_cluster_length), "-M", "0", "-d", "0"
    ], check=True, capture_output=True, text=True)

    cluster_map = {}
    current_cluster = None
    with open(output_prefix + ".clstr") as f:
        for line in f:
            if line.startswith(">Cluster"):
                current_cluster = int(line.strip().split()[-1])
            else:
                m = re.search(r">seq(\d+)\.\.\.", line)
                if m:
                    cluster_map[int(m.group(1))] = current_cluster

    next_id = max(cluster_map.values(), default=-1) + 1
    for i in df[short_mask].index:
        cluster_map[i] = next_id
        next_id += 1

    df["cluster_40"] = df.index.map(cluster_map)
    assert df["cluster_40"].isna().sum() == 0
    return df

aop_benchpos = cluster_40_safe(aop_benchpos)
print("Sequences:", len(aop_benchpos), "| 40%-identity clusters:", aop_benchpos["cluster_40"].nunique())
print("Missing cluster assignments:", aop_benchpos["cluster_40"].isna().sum())

import random
random.seed(42)

def cluster_safe_kfold(df, n_folds=2):
    clusters = df["cluster_40"].unique().tolist()
    random.shuffle(clusters)
    fold_of_cluster = {c: i % n_folds for i, c in enumerate(clusters)}
    return df["cluster_40"].map(fold_of_cluster)

tier12_mask = aop_benchpos["tier"].isin(["Tier1", "Tier2"])
aop_benchpos.loc[tier12_mask, "lso_fold"] = cluster_safe_kfold(aop_benchpos[tier12_mask], n_folds=2)

print(aop_benchpos[tier12_mask].groupby(["tier", "lso_fold"]).size())

# For a direct like-for-like comparison against GraphPart (which split ALL tiers, not just Tier1/2):
aop_benchpos["cluster_40_fold_all"] = cluster_safe_kfold(aop_benchpos, n_folds=2)
print(aop_benchpos.groupby(["tier", "cluster_40_fold_all"]).size())

Sequences: 1639 | 40%-identity clusters: 1483
Missing cluster assignments: 0
tier   lso_fold
Tier1  0.0         315
       1.0         316
Tier2  0.0          74
       1.0          76
dtype: int64
tier   cluster_40_fold_all
Tier1  0                      329
       1                      302
Tier2  0                       81
       1                       69
Tier3  0                      418
       1                      440
dtype: int64
